# Impulse — A2D2 Analysis: events & event GIFs

Companion to **`a2d2_ingestion.ipynb`**. This notebook reads only the already-ingested
tables/images (it downloads nothing and commits no data) and demonstrates ad-hoc **TSAL**
analysis on the A2D2 bus signals, then the headline feature: **detect driving events with
Impulse, persist them to the gold `event_instance_fact`, and visualize each event as an
animated GIF** stitched from the camera frames in that event's time window.

## Prerequisite
Run **`a2d2_ingestion.ipynb` first**, ideally with a higher `images_per_second` (e.g. a few
frames/s) so each short event window contains enough frames for a smooth GIF. This notebook
expects `{catalog}.{schema}.{table_prefix}_*`: the 5 silver tables and `*_camera_frames`
(the timestamp → image-path mapping, with the PNGs on the volume).

## License
A2D2 © Audi AG, **CC BY-ND 4.0** (https://creativecommons.org/licenses/by-nd/4.0/, source
https://www.a2d2.audi/). This notebook reads only your own ingested tables/images — it
redistributes nothing. Commit it with outputs cleared.

In [ ]:
%pip install pydantic>=2.0 scipy matplotlib pillow -q
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")
dbutils.widgets.text("table_prefix", "a2d2", "Table Prefix")
dbutils.widgets.text("volume", "a2d2_raw", "UC Volume")
dbutils.widgets.text("gif_max_duration_s", "5", "GIF max duration (s)")
dbutils.widgets.text("gif_play_fps", "5", "GIF playback fps")

In [ ]:
import sys, os

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
TABLE_PREFIX = dbutils.widgets.get("table_prefix")
VOLUME = dbutils.widgets.get("volume")
GIF_MAX_DURATION_S = float(dbutils.widgets.get("gif_max_duration_s") or "5")
GIF_PLAY_FPS = float(dbutils.widgets.get("gif_play_fps") or "5")

if not (CATALOG and SCHEMA and TABLE_PREFIX):
    raise ValueError("Please set catalog, schema and table_prefix widgets above.")

# Works from a cloned repo/Git folder (src on path) and as a DABs wheel-installed job.
nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = "/Workspace" + "/".join(nb_path.split("/")[:-3])
src_dir = os.path.join(REPO_ROOT, "src")
if os.path.isdir(src_dir):
    sys.path.insert(0, src_dir)

pfx = f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}"
CONTAINER_ID = 1
if not spark.catalog.tableExists(f"{pfx}_channels"):
    raise RuntimeError(f"{pfx}_channels not found. Run a2d2_ingestion.ipynb first.")
print("Reading from", f"{pfx}_*")

In [ ]:
from databricks.sdk import WorkspaceClient
from impulse_reporting.core.report import Report
from impulse_reporting.core.page import Page
from impulse_reporting.events.basic_event import BasicEvent
from impulse_reporting.events.container_event import ContainerEvent
from impulse_reporting.aggregations.histogram2d import Histogram2DDuration
from impulse_reporting.aggregations.stats_aggregator import StatsAggregator
import numpy as np
import pandas as pd
import pyspark.sql.functions as F

config = {
    "source": {
        "container_metrics_table": f"{pfx}_container_metrics",
        "container_tags_table": f"{pfx}_container_tags",
        "channel_metrics_table": f"{pfx}_channel_metrics",
        "channel_tags_table": f"{pfx}_channel_tags",
        "channels_uri": f"{pfx}_channels",
    },
    "unity_sink": {"catalog": CATALOG, "schema": SCHEMA, "table_prefix": TABLE_PREFIX},
    "query_engine": {"solver": "DeltaSolver", "data_type": "RAW"},
    "measurement_dimensions": ["container_id"],
}
report = Report(name="a2d2_analysis", spark=spark, workspace_client=WorkspaceClient(), config=config)
db = report.get_db()
solver = report.get_solver()

# Read channel name -> unit (for chart labels) instead of hardcoding.
_tags = spark.read.table(f"{pfx}_channel_tags")
_names = _tags.where("key = 'channel_name'").select("channel_id", "value").toPandas()
_units = _tags.where("key = 'unit'").select("channel_id", "value").toPandas()
_merged = _names.merge(_units, on="channel_id", how="left", suffixes=("_name", "_unit"))
unit_of = dict(zip(_merged["value_name"], _merged["value_unit"]))
print("channel units:", unit_of)

## 1. Physical & virtual channels (ad-hoc TSAL)

Select channels by tag and derive virtual signals, then solve once to pandas. The query
engine returns one row per container; each selected channel comes back as a `SampleSeries`.

In [ ]:
speed = db.query.channel(channel_name="vehicle_speed")
acc_x = db.query.channel(channel_name="acceleration_x")
acc_y = db.query.channel(channel_name="acceleration_y")
steer = db.query.channel(channel_name="steering_angle_calculated")
brake = db.query.channel(channel_name="brake_pressure")

# Virtual signal: cumulative distance (km) from speed (km/h). RAW timestamps are in us,
# so resample to 1 s (1e6 us) before integrating.
distance_km = speed.resample(1e6).cumtrapz() / 3600.0 / 1e6

pdf = (db.query
       .select(speed.alias("speed"), acc_x.alias("acc_x"), acc_y.alias("acc_y"),
               steer.alias("steer"), brake.alias("brake"), distance_km.alias("distance_km"))
       .toPandas(spark, solver=solver))
row = pdf.iloc[0]
for k in ["speed", "acc_x", "acc_y", "steer", "brake", "distance_km"]:
    ss = row[k]
    print(f"{k:11s} n={len(ss.values):>7,}  min={ss.min():.2f}  max={ss.max():.2f}")
try:
    print(f"total distance ~ {row['distance_km'].values[-1]:.2f} km")
except Exception as e:
    print("distance check skipped:", e)

In [ ]:
try:
    import matplotlib.pyplot as plt
    sp = row["speed"]
    t0 = sp.tstarts[0]
    def secs(ss):
        return (ss.tstarts - t0) / 1e6
    fig, ax = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
    ax[0].plot(secs(row["speed"]), row["speed"].values); ax[0].set_ylabel("speed km/h")
    ax[1].plot(secs(row["acc_x"]), row["acc_x"].values, color="tab:red"); ax[1].set_ylabel("accel_x m/s^2")
    ax[2].plot(secs(row["steer"]), row["steer"].values, color="tab:green"); ax[2].set_ylabel("steering deg")
    ax[3].plot(secs(row["distance_km"]), row["distance_km"].values, color="tab:purple"); ax[3].set_ylabel("distance km")
    ax[3].set_xlabel("seconds from start of recording")
    fig.suptitle("A2D2 bus signals — recording 20190401_121727"); plt.tight_layout(); plt.show()
except Exception as e:
    print("plot skipped:", e)
    display(spark.read.table(f"{pfx}_channel_metrics"))

## 2. Events

Define driving events as TSAL boolean expressions (thresholds are tunable for the
recording). Every registered event is computed and persisted to `event_instance_fact`.

In [ ]:
# Thresholds tuned to surface a handful of instances; adjust for your recording.
hard_brake_expr  = (brake > 20.0) | (acc_x < -2.0)     # high brake pressure OR strong deceleration
sharp_steer_expr = (steer > 45.0) | (steer < -45.0)    # |steering| > 45 deg
high_speed_expr  = speed > 50.0                         # > 50 km/h

ev_brake = BasicEvent(name="hard_braking",  expr=hard_brake_expr,  desc="brake_pressure>20 or accel_x<-2")
ev_steer = BasicEvent(name="sharp_steering", expr=sharp_steer_expr, desc="|steering|>45 deg")
ev_speed = BasicEvent(name="high_speed",    expr=high_speed_expr,  desc="vehicle_speed>50 km/h")
ev_drive = ContainerEvent(name="whole_drive", desc="entire recording")
for e in (ev_brake, ev_steer, ev_speed, ev_drive):
    report.add_event(e)
print("registered events:", [e.get_name() for e in report.get_events()])

## 3. Aggregations — speed × lateral-acceleration heatmap + event-scoped stats

In [ ]:
# Quick in-memory 2D histogram (no Spark) directly from the SampleSeries.
try:
    import matplotlib.pyplot as plt
    sp, ay = row["speed"].synchronized(row["acc_y"])
    x_bins = [float(x) for x in range(0, 141, 10)]
    y_bins = [round(-4 + 0.5 * i, 1) for i in range(17)]
    H, xe, ye = sp.histogram2d(ay, x_bins, y_bins)
    plt.figure(figsize=(9, 5))
    plt.pcolormesh(xe, ye, H.T)
    plt.colorbar(label="duration (s-weighted)")
    plt.xlabel("vehicle_speed km/h"); plt.ylabel("acceleration_y m/s^2")
    plt.title("Time spent in speed x lateral-accel bins"); plt.show()
except Exception as e:
    print("2D histogram preview skipped:", e)

In [ ]:
# Persist gold-layer aggregations via the Report pipeline. This also computes and
# persists every registered event into {pfx}_event_instance_fact.
page = Page(page_number=1)
page.add_aggregation(Histogram2DDuration(
    name="a2d2_speed_vs_lat_accel",
    x_expr=speed, y_expr=acc_y,
    x_bins=[float(x) for x in range(0, 141, 10)],
    y_bins=[float(y) / 2.0 for y in range(-8, 9)],
    event=ev_drive,
    x_channel_name="vehicle_speed", y_channel_name="acceleration_y",
    x_bins_unit="km/h", y_bins_unit="m/s^2", values_unit="s",
))
page.add_aggregation(StatsAggregator(
    name="a2d2_braking_stats",
    input_expressions=[speed, acc_x, brake, steer],
    channel_names=["vehicle_speed", "acceleration_x", "brake_pressure", "steering_angle_calculated"],
    statistics=["min", "mean", "max"],
    event=ev_brake,
))
report.add_page(page)
report.determine_report()
report.persist_results()

print("Event instances per event:")
display(
    spark.read.table(f"{pfx}_event_instance_fact").alias("f")
    .join(spark.read.table(f"{pfx}_event_dimension").select("event_id", "event_name").alias("d"), "event_id")
    .groupBy("event_name").count().orderBy(F.desc("count"))
)

## 4. Visualize events as GIFs

For each interesting event instance we take the camera frames whose `cam_tstamp` falls in
the instance window (capped at `gif_max_duration_s`), stitch them into an animated GIF on
the driver under `/tmp/a2d2_gifs`, and show it inline.

In [ ]:
import base64
from PIL import Image

def frames_to_gif(image_paths, out_path, frame_ms=200, loop=0, width=480):
    """Build an animated GIF from local image files. frame_ms: int or per-frame list."""
    imgs = []
    for p in image_paths:
        im = Image.open(p).convert("RGB")
        if width:
            w, h = im.size
            im = im.resize((width, max(1, int(h * width / w))))
        imgs.append(im)
    if not imgs:
        raise ValueError("no frames")
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    imgs[0].save(out_path, save_all=True, append_images=imgs[1:],
                 duration=frame_ms, loop=loop, optimize=True)
    return out_path

def event_clip_frames(frames_pdf, start_ts, end_ts, max_duration_s=5.0, max_frames=60):
    """Frames within [start_ts, min(end_ts, start_ts+max_duration)] (us), capped to max_frames."""
    end = min(int(end_ts), int(start_ts) + int(max_duration_s * 1e6))
    sub = frames_pdf[(frames_pdf.cam_tstamp_us >= start_ts) & (frames_pdf.cam_tstamp_us <= end)]
    sub = sub.sort_values("cam_tstamp_us")
    if len(sub) > max_frames:
        idx = np.linspace(0, len(sub) - 1, max_frames).round().astype(int)
        sub = sub.iloc[idx]
    return sub

def event_gif(name, instance_id, sub_pdf, out_dir="/tmp/a2d2_gifs", play_fps=5.0, width=480):
    out = f"{out_dir}/{name}_{int(instance_id)}.gif"
    return frames_to_gif(sub_pdf.png_path.tolist(), out, frame_ms=int(1000 / play_fps), width=width)

def show_gif(path, width=480):
    with open(path, "rb") as f:
        b = base64.b64encode(f.read()).decode()
    displayHTML(f'<p><b>{os.path.basename(path)}</b></p>'
                f'<img src="data:image/gif;base64,{b}" width="{width}"/>')

In [ ]:
if not spark.catalog.tableExists(f"{pfx}_camera_frames"):
    print(f"{pfx}_camera_frames not found - re-run ingestion with download_images=true.")
else:
    frames_pdf = (spark.read.table(f"{pfx}_camera_frames")
                  .select("cam_tstamp_us", "png_path").orderBy("cam_tstamp_us").toPandas())
    ev = (spark.read.table(f"{pfx}_event_instance_fact")
          .join(spark.read.table(f"{pfx}_event_dimension").select("event_id", "event_name"), "event_id"))
    counts = ev.groupBy("event_name").count().orderBy(F.desc("count")).toPandas()

    if counts.empty or frames_pdf.empty:
        print("No event instances or no camera frames available.")
    else:
        chosen = counts.iloc[0]["event_name"]
        print(f"Visualizing event '{chosen}' ({int(counts.iloc[0]['count'])} instances). "
              f"{len(frames_pdf)} camera frames available.")
        inst = (ev.where(F.col("event_name") == chosen)
                .select("event_instance_id", "start_ts", "end_ts").orderBy("start_ts").limit(3).toPandas())
        shown = 0
        for _, r in inst.iterrows():
            sub = event_clip_frames(frames_pdf, r["start_ts"], r["end_ts"], GIF_MAX_DURATION_S)
            t0 = pd.to_datetime(int(r["start_ts"]) * 1000)  # us -> ns
            if len(sub) == 0:
                print(f"  instance {int(r['event_instance_id'])} @ {t0} UTC: 0 frames in window "
                      f"(increase images_per_second at ingestion for denser clips).")
                continue
            path = event_gif(chosen, r["event_instance_id"], sub, play_fps=GIF_PLAY_FPS)
            print(f"  instance {int(r['event_instance_id'])} @ {t0} UTC: {len(sub)} frames -> {path}")
            show_gif(path)
            shown += 1
        if shown == 0:
            print("No instance had >=1 frame in its window. Ingest more images "
                  "(higher images_per_second) and/or raise gif_max_duration_s.")

## Notes
- GIFs are written to `/tmp/a2d2_gifs` on the driver (ephemeral; not committed, not in UC).
- This notebook persisted new gold tables (`_histogram2d_*`, `_stats_aggregator_*`,
  `_event_*`); the ingestion notebook's `drop_created_tables` widget removes all
  `{prefix}_*` tables.
- For smooth, real-time-looking clips, ingest at a higher `images_per_second` so each event
  window holds several frames.